# Disaster Risk Data: US Per-State CSV Generator

Generates per-state CSV files with FEMA NRI county-level hazard risk scores
for all 50 US states and DC. Data source: **[FEMA National Risk Index (NRI)](https://www.fema.gov/about/openfema/data-sets/national-risk-index-data)**:
  18 natural-hazard risk scores for every US county (December 2025, v3).

See `CA-MX_disaster_risk_analysis.ipynb` for Canadian and Mexican equivalents (mocked).

## Output

Per-state CSV files written to `../plugins/emfn-behavior-plugin/assets/data/`:

| Region | Files | Source |
|--------|-------|--------|
| US states + DC | `AL.csv` … `WY.csv` (51 files) | NRI real data |

Each file has columns: `county_fips`, `state`, `county`, `{HAZARD}_risk_score` × 18.


In [ ]:
# Install all project dependencies declared in pyproject.toml into the venv.
# Skip if you have already run `uv sync` from the repo root.
!uv sync

In [ ]:
import tempfile, zipfile
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

## US — FEMA National Risk Index

The NRI ZIP (~50 MB) is downloaded once and cached at `cache/NRI_Table_Counties.csv`.
Re-running uses the cached file. Delete the cache file to force a fresh download.

In [ ]:
# Paths
NRI_ZIP_URL   = "https://www.fema.gov/about/reports-and-data/openfema/nri/v120/NRI_Table_Counties.zip"
NRI_CSV_CACHE = Path("cache/NRI_Table_Counties.csv")
OUTPUT_DIR    = Path("../plugins/emfn-behavior-plugin/assets/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NRI hazard codes (December 2025, v3) — order matches AL.csv sample
HAZARD_CODES = [
    "AVLN", "CFLD", "CWAV", "DRGT", "ERQK", "HAIL",
    "HWAV", "HRCN", "ISTM", "LNDS", "LTNG", "IFLD",
    "SWND", "TRND", "TSUN", "VLCN", "WFIR", "WNTW",
]

# USPS 2-letter state codes (50 states + DC)
US_STATES = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

print(f"US states: {len(US_STATES)}")
print(f"Hazard types: {len(HAZARD_CODES)}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")


In [ ]:
def load_nri_data() -> pd.DataFrame:
    """Return the NRI county table, downloading from FEMA and caching locally on first run."""
    if NRI_CSV_CACHE.exists():
        print(f"Loading cached NRI data from {NRI_CSV_CACHE} …")
        return pd.read_csv(NRI_CSV_CACHE, dtype={"STCOFIPS": str})

    print(f"Downloading NRI data from FEMA …")
    NRI_CSV_CACHE.parent.mkdir(parents=True, exist_ok=True)

    temp_zip_path = None
    with requests.get(NRI_ZIP_URL, stream=True, timeout=300) as resp:
        resp.raise_for_status()

        total = int(resp.headers.get("content-length", 0))
        try:
            with tempfile.NamedTemporaryFile(
                suffix=".zip",
                delete=False,
                dir=NRI_CSV_CACHE.parent,
            ) as tmp_zip:
                temp_zip_path = Path(tmp_zip.name)
                with tqdm(total=total, unit="B", unit_scale=True, desc="Downloading") as bar:
                    for chunk in resp.iter_content(chunk_size=65536):
                        if not chunk:
                            continue
                        tmp_zip.write(chunk)
                        bar.update(len(chunk))

            with zipfile.ZipFile(temp_zip_path) as zf:
                csv_name = next(
                    n for n in zf.namelist()
                    if n.endswith(".csv") and "NRI_Table_Counties" in n
                )
                print(f"Extracting {csv_name} …")
                with zf.open(csv_name) as f:
                    df = pd.read_csv(f, dtype={"STCOFIPS": str})
        finally:
            if temp_zip_path and temp_zip_path.exists():
                temp_zip_path.unlink()

    df.to_csv(NRI_CSV_CACHE, index=False)
    print(f"Cached to {NRI_CSV_CACHE}")
    return df


nri_raw = load_nri_data()
print(f"\nLoaded {len(nri_raw):,} counties · {nri_raw.shape[1]} columns")
print("State column candidates:", [c for c in nri_raw.columns if "STATE" in c.upper()])

In [ ]:
def nri_rows_to_output(df: pd.DataFrame) -> pd.DataFrame:
    """Convert a slice of the raw NRI table to the output CSV schema.

    Output columns: county_fips, state, county, {HAZARD}_risk_score × 18.
    Handles the RFLD→IFLD rename between NRI v2 and v3.
    """
    out = pd.DataFrame({
        "county_fips": df["STCOFIPS"],
        "state":       df["STATE"],
        "county":      df["COUNTY"],
    })
    for haz in HAZARD_CODES:
        src = f"{haz}_RISKS"
        if haz == "IFLD" and src not in df.columns and "RFLD_RISKS" in df.columns:
            src = "RFLD_RISKS"   # NRI v2 used RFLD; v3 uses IFLD
        out[f"{haz}_risk_score"] = df[src].values if src in df.columns else pd.NA
    return out.reset_index(drop=True)


# Identify the state-abbreviation column (varies by NRI version)
abbr_col = next(
    (c for c in ("STATEABBRV", "STUSPS", "STATE_ABBRV") if c in nri_raw.columns),
    None,
)
if abbr_col is None:
    nri_raw = nri_raw.copy()
    nri_raw["_ABBR"] = nri_raw["STATE"].map(US_STATES)
    abbr_col = "_ABBR"

written = 0
valid_abbrs = set(US_STATES.values())
for abbr, group in tqdm(nri_raw.groupby(abbr_col), desc="Writing US state CSVs"):
    if pd.isna(abbr) or abbr not in valid_abbrs:
        continue
    out_df = nri_rows_to_output(group)
    (OUTPUT_DIR / f"{abbr}.csv").write_text(out_df.to_csv(index=False))
    written += 1

print(f"\nWrote {written} US state CSVs to {OUTPUT_DIR}/")

## Summary

In [ ]:
all_csvs = sorted(OUTPUT_DIR.glob("*.csv"))
us = [f for f in all_csvs if f.stem in set(US_STATES.values())]

print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"  US state CSVs: {len(us):3d}  ({', '.join(f.stem for f in us[:6])} …)")


### Integration tests

Validates that every generated CSV satisfies the schema contract expected by the JS client (`emfn-behavior-plugin.js`).

| Test | What it checks |
|------|---------------|
| T1 | All 51 state CSVs exist |
| T2 | Column names match the JS-expected set exactly (`county_fips`, `state`, `county`, 18 × `{HAZ}_risk_score`) |
| T3 | `county_fips` is always a 5-digit zero-padded string (required for the JS FIPS lookup) |
| T4 | No duplicate `county_fips` within a single state file |
| T5 | All hazard score columns are numeric or blank — no stray strings that would make `parseFloat` return `NaN` |
| T6 | Present scores are within the valid `[0, 100]` percentile range |

In [ ]:
import gc
import re as _re

# Hazard codes the JS client maps via NRI_HAZARD_LABELS — order must match emfn-behavior-plugin.js
JS_HAZARD_CODES = [
    "AVLN", "CFLD", "CWAV", "DRGT", "ERQK", "HAIL",
    "HWAV", "HRCN", "ISTM", "LNDS", "LTNG", "IFLD",
    "SWND", "TRND", "TSUN", "VLCN", "WFIR", "WNTW",
]
EXPECTED_COLS  = ["county_fips", "state", "county"] + [f"{h}_risk_score" for h in JS_HAZARD_CODES]
FIPS_RE        = _re.compile(r"^\d{5}$")
EXPECTED_ABBRS = set(US_STATES.values())

failures = []

# T1 – every expected state CSV exists
missing_files = EXPECTED_ABBRS - {f.stem for f in all_csvs}
if missing_files:
    failures.append(f"T1 FAIL – missing CSV(s): {sorted(missing_files)}")
else:
    print(f"T1 PASS  all {len(EXPECTED_ABBRS)} state CSVs present")

# T2–T6 – per-file schema and data integrity
for csv_path in us:
    abbr = csv_path.stem
    try:
        df = pd.read_csv(csv_path, dtype={"county_fips": str})
    except Exception as exc:
        failures.append(f"T2 FAIL [{abbr}] unreadable: {exc}")
        continue

    # T2 – column names match JS-expected schema
    missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing_cols:
        failures.append(f"T2 FAIL [{abbr}] missing columns: {missing_cols}")

    # T3 – county_fips is always a 5-digit zero-padded string
    present  = df["county_fips"].dropna()
    bad_fips = present[~present.str.match(FIPS_RE)]
    if not bad_fips.empty:
        failures.append(f"T3 FAIL [{abbr}] malformed county_fips: {bad_fips.tolist()[:5]}")

    # T4 – no duplicate FIPS within a state file
    dupes = df.loc[df["county_fips"].duplicated(), "county_fips"]
    if not dupes.empty:
        failures.append(f"T4 FAIL [{abbr}] duplicate county_fips: {dupes.tolist()}")

    # T5 + T6 – score columns: numeric-or-blank, within [0, 100]
    score_cols = [c for c in EXPECTED_COLS if c.endswith("_risk_score") and c in df.columns]
    for col in score_cols:
        vals    = df[col].dropna()                       # non-null values, intact index
        numeric = pd.to_numeric(vals, errors="coerce")   # same index; NaN where non-numeric
        bad_str = vals[numeric.isna()]                   # T5: entries that couldn't be parsed
        if not bad_str.empty:
            failures.append(f"T5 FAIL [{abbr}][{col}] non-numeric: {bad_str.tolist()[:3]}")
        oor = numeric[(numeric < 0) | (numeric > 100)]   # T6: out-of-range scores
        if not oor.empty:
            failures.append(f"T6 FAIL [{abbr}][{col}] out-of-range scores: {oor.tolist()[:3]}")

    del df
    gc.collect()  # release each DataFrame's cyclic refs synchronously

# ── Report ──────────────────────────────────────────────────────────────────────────────────
print()
if failures:
    for msg in failures:
        print(msg)
    raise AssertionError(f"{len(failures)} integration test failure(s) — see above")
else:
    print(f"✅  All {len(us)} state CSVs passed T1–T6")
